Data from : https://all-that-glitters-is-not-gold-123.blogspot.com/2025/11/am-men-catalog-log-legend-n-normal.html



In [10]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable, List, Optional, Set, Dict
import re

# ============================================================
# 0) Philosophy
# ------------------------------------------------------------
# - This engine is for "weed-out": unsafe / not-normal / insane men.
# - Each rule outputs an ABNORMALITY SCORE (1..10).
# - Total risk score is the SUM of fired-rule scores (0..N*10).
# - Some rules are HARD_EXIT (immediate BLOCK).
# - Non-negotiables are evaluated separately as "alignment mismatch"
#   (safe person can still be a mismatch).
# ============================================================


# ============================================================
# 1) Your Non-Negotiables (edit here)
# ============================================================

@dataclass
class NonNegotiables:
    # Core values
    partner_must_be_virgin: bool = True
    no_smoking: bool = True
    no_addictions: bool = True  # alcohol/drugs
    wants_kids_required: bool = True
    not_atheist: bool = True

    # NEW: vegetarian + Brahmin + South India
    must_be_vegetarian: bool = True
    must_be_brahmin: bool = True
    must_be_south_indian: bool = True

    # Safety hard lines
    zero_tolerance_stalking: bool = True
    zero_tolerance_catfishing: bool = True
    zero_tolerance_sexual_coercion: bool = True
    zero_tolerance_misogyny: bool = True

    # Optional constraint
    max_age_diff_years: Optional[int] = 2


# ============================================================
# 2) Data model for each match
# ============================================================

@dataclass
class Prospect:
    id: str
    alias: str = ""

    key_phrases: List[str] = field(default_factory=list)  # quotes / his phrases
    notes: str = ""                                       # your summary
    behaviors: Set[str] = field(default_factory=set)      # normalized tokens

    # Facts/claims (True/False/None)
    is_virgin: Optional[bool] = None
    smoked: Optional[bool] = None
    drinks: Optional[bool] = None
    drugs: Optional[bool] = None
    wants_kids: Optional[bool] = None
    atheist: Optional[bool] = None
    age_diff_years: Optional[int] = None

    vegetarian: Optional[bool] = None
    brahmin: Optional[bool] = None
    south_indian: Optional[bool] = None

    attraction: Optional[str] = None  # "attracted" / "not_attracted" / None


# ============================================================
# 3) Rule engine structures
# ============================================================

@dataclass
class RuleHit:
    rule_id: str
    abnormality: int              # 1..10
    severity: str                 # HARD_EXIT | HIGH | MEDIUM | LOW | INFO
    tags: List[str]
    reason: str


@dataclass
class Decision:
    category: str                 # UNSAFE | NOT_NORMAL | NORMAL_MISMATCH | NORMAL_POSSIBLE
    decision: str                 # BLOCK | REJECT | CAUTION | PASS
    abnormality_total: int        # sum of fired abnormality scores
    abnormality_level: str        # low/medium/high/extreme
    nonneg_mismatch: bool
    unknown_nonneg_fields: List[str]
    tags: List[str]
    fired_rules: List[RuleHit]


@dataclass
class Rule:
    rule_id: str
    severity: str
    abnormality: int              # 1..10
    tags: List[str]
    condition: Callable[[Prospect, NonNegotiables], bool]
    reason: str


# ============================================================
# 4) Helper matchers
# ============================================================

def has_behavior(p: Prospect, *tokens: str) -> bool:
    lower = {b.lower() for b in p.behaviors}
    return any(t.lower() in lower for t in tokens)

def phrase_matches(p: Prospect, patterns: List[str]) -> bool:
    text = " | ".join(p.key_phrases + [p.notes]).lower()
    return any(re.search(ptn, text) for ptn in patterns)

def clamp_1_10(x: int) -> int:
    return max(1, min(10, x))


# ============================================================
# 5) The Engine
# ============================================================

class WeedOutEngine:
    def __init__(
        self,
        rules: List[Rule],
        hard_exit_severity: str = "HARD_EXIT",
        not_normal_threshold: int = 18,  # total abnormality points
        unsafe_threshold: int = 26       # total abnormality points
    ):
        self.rules = rules
        self.hard_exit_severity = hard_exit_severity
        self.not_normal_threshold = not_normal_threshold
        self.unsafe_threshold = unsafe_threshold

    def evaluate(self, p: Prospect, nn: NonNegotiables) -> Decision:
        hits: List[RuleHit] = []
        tagset: Set[str] = set()
        total = 0

        # Fire rules
        for r in self.rules:
            if r.condition(p, nn):
                abn = clamp_1_10(r.abnormality)
                total += abn
                tagset.update(r.tags)
                hits.append(RuleHit(
                    rule_id=r.rule_id,
                    abnormality=abn,
                    severity=r.severity,
                    tags=r.tags,
                    reason=r.reason
                ))

        # Hard exit gate
        if any(h.severity == self.hard_exit_severity for h in hits):
            mismatch = True
            unknowns = self._unknown_nonneg_fields(p, nn)
            return Decision(
                category="UNSAFE",
                decision="BLOCK",
                abnormality_total=max(total, self.unsafe_threshold),
                abnormality_level=self._level(max(total, self.unsafe_threshold)),
                nonneg_mismatch=mismatch,
                unknown_nonneg_fields=unknowns,
                tags=sorted(tagset),
                fired_rules=hits
            )

        # Non-neg mismatch (alignment) is separate from abnormality
        mismatch = self._nonneg_mismatch(p, nn)
        unknowns = self._unknown_nonneg_fields(p, nn)

        # Category by abnormality
        if total >= self.unsafe_threshold:
            category = "UNSAFE"
            decision_str = "BLOCK"
        elif total >= self.not_normal_threshold:
            category = "NOT_NORMAL"
            decision_str = "REJECT"
        else:
            if mismatch:
                category = "NORMAL_MISMATCH"
                decision_str = "REJECT"
            else:
                category = "NORMAL_POSSIBLE"
                decision_str = "PASS"

        # If still PASS but unknown key non-negs exist => CAUTION
        if decision_str == "PASS" and unknowns:
            decision_str = "CAUTION"
            tagset.add("verify-nonneg")

        # If PASS but has watch tags => CAUTION
        if decision_str == "PASS" and any(t in tagset for t in ["watch", "soft-risk", "inconsistency"]):
            decision_str = "CAUTION"

        return Decision(
            category=category,
            decision=decision_str,
            abnormality_total=total,
            abnormality_level=self._level(total),
            nonneg_mismatch=mismatch,
            unknown_nonneg_fields=unknowns,
            tags=sorted(tagset),
            fired_rules=sorted(hits, key=lambda h: (-h.abnormality, h.rule_id))
        )

    def _level(self, total: int) -> str:
        if total >= 30:
            return "extreme"
        if total >= 20:
            return "high"
        if total >= 10:
            return "medium"
        return "low"

    def _unknown_nonneg_fields(self, p: Prospect, nn: NonNegotiables) -> List[str]:
        unknowns = []

        # Only include as unknown if it's required by your NN settings
        if nn.partner_must_be_virgin and p.is_virgin is None:
            unknowns.append("is_virgin")
        if nn.no_smoking and p.smoked is None:
            unknowns.append("smoked")
        if nn.no_addictions and (p.drinks is None and p.drugs is None):
            unknowns.append("drinks/drugs")
        if nn.wants_kids_required and p.wants_kids is None:
            unknowns.append("wants_kids")
        if nn.not_atheist and p.atheist is None:
            unknowns.append("atheist")
        if nn.must_be_vegetarian and p.vegetarian is None:
            unknowns.append("vegetarian")
        if nn.must_be_brahmin and p.brahmin is None:
            unknowns.append("brahmin")
        if nn.must_be_south_indian and p.south_indian is None:
            unknowns.append("south_indian")
        return unknowns

    def _nonneg_mismatch(self, p: Prospect, nn: NonNegotiables) -> bool:
        # Unknown does not count as mismatch; it counts as "verify"
        if nn.partner_must_be_virgin and p.is_virgin is False:
            return True
        if nn.no_smoking and p.smoked is True:
            return True
        if nn.no_addictions and (p.drinks is True or p.drugs is True):
            return True
        if nn.wants_kids_required and p.wants_kids is False:
            return True
        if nn.not_atheist and p.atheist is True:
            return True

        if nn.must_be_vegetarian and p.vegetarian is False:
            return True
        if nn.must_be_brahmin and p.brahmin is False:
            return True
        if nn.must_be_south_indian and p.south_indian is False:
            return True

        if nn.max_age_diff_years is not None and p.age_diff_years is not None:
            if abs(p.age_diff_years) > nn.max_age_diff_years:
                return True

        return False


# ============================================================
# 6) Ruleset (built from your patterns)
#    Each rule abnormality score is 1..10.
# ============================================================

def build_rules() -> List[Rule]:
    rules: List[Rule] = []

    # ---------------- HARD EXIT (UNSAFE) ----------------
    rules.append(Rule(
        rule_id="HARD_STALKING",
        severity="HARD_EXIT",
        abnormality=10,
        tags=["safety", "stalking"],
        condition=lambda p, nn: nn.zero_tolerance_stalking and has_behavior(p, "stalking", "doxxing"),
        reason="Stalking/doxxing is an immediate safety exit."
    ))

    rules.append(Rule(
        rule_id="HARD_CATFISHING_IDENTITY_AGE",
        severity="HARD_EXIT",
        abnormality=10,
        tags=["safety", "deception", "catfishing"],
        condition=lambda p, nn: nn.zero_tolerance_catfishing and has_behavior(p, "catfishing", "identity_mismatch", "age_manipulation"),
        reason="Identity/age manipulation/catfishing is an immediate safety exit."
    ))

    rules.append(Rule(
        rule_id="HARD_SEXUAL_COERCION_NUDES",
        severity="HARD_EXIT",
        abnormality=10,
        tags=["safety", "sexual-pressure"],
        condition=lambda p, nn: nn.zero_tolerance_sexual_coercion and has_behavior(p, "asked_for_nudes", "sexual_pressure", "explicit_sex_talk"),
        reason="Nudes request / sexual coercion / explicit sex talk early is an immediate exit."
    ))

    rules.append(Rule(
        rule_id="HARD_THREATS_INTIMIDATION",
        severity="HARD_EXIT",
        abnormality=10,
        tags=["safety", "threats"],
        condition=lambda p, nn: has_behavior(p, "threatening", "intimidation"),
        reason="Threats/intimidation is an immediate exit."
    ))

    rules.append(Rule(
        rule_id="HARD_MISOGYNY",
        severity="HARD_EXIT",
        abnormality=10,
        tags=["safety", "misogyny"],
        condition=lambda p, nn: nn.zero_tolerance_misogyny and (has_behavior(p, "misogyny") or phrase_matches(p, [r"all women are bad", r"women are evil"])),
        reason="Misogyny/contempt toward women is unsafe and an immediate exit."
    ))

    # ---------------- HIGH RISK (NOT NORMAL) ----------------
    rules.append(Rule(
        rule_id="HIGH_GHOSTING_RETURNING",
        severity="HIGH",
        abnormality=8,
        tags=["inconsistency", "reliability"],
        condition=lambda p, nn: has_behavior(p, "ghosting") and has_behavior(p, "returned_after_ghosting"),
        reason="Ghosting + returning suggests hot-cold cycles and unreliability."
    ))

    rules.append(Rule(
        rule_id="HIGH_GHOSTING",
        severity="HIGH",
        abnormality=7,
        tags=["inconsistency", "reliability"],
        condition=lambda p, nn: has_behavior(p, "ghosting"),
        reason="Ghosting indicates poor communication and low reliability."
    ))

    rules.append(Rule(
        rule_id="HIGH_DODGE_NONNEG_DIRECT",
        severity="HIGH",
        abnormality=8,
        tags=["integrity", "nonneg-evasion"],
        condition=lambda p, nn: has_behavior(p, "dodged_nonneg") or phrase_matches(p, [r"\bdeflect", r"\bavoid", r"went silent", r"changed the topic"]),
        reason="Dodging non-negotiable questions is a strong integrity risk."
    ))

    rules.append(Rule(
        rule_id="HIGH_ENTITLEMENT_QUEUE_LANGUAGE",
        severity="HIGH",
        abnormality=8,
        tags=["entitlement", "objectification"],
        condition=lambda p, nn: phrase_matches(p, [r"queue", r"line of girls", r"waiting for women"]),
        reason="Entitlement/objectification language about women is a strong character red flag."
    ))

    rules.append(Rule(
        rule_id="HIGH_HELP_AS_LEVERAGE",
        severity="HIGH",
        abnormality=7,
        tags=["control", "power-dynamic"],
        condition=lambda p, nn: has_behavior(p, "help_as_leverage") or phrase_matches(p, [r"helping because you('re| are) a girl", r"you need help"]),
        reason="Help framed as gender/power can become leverage/control."
    ))

    rules.append(Rule(
        rule_id="HIGH_BOUNDARY_PUSHING",
        severity="HIGH",
        abnormality=8,
        tags=["boundary-violation", "control"],
        condition=lambda p, nn: has_behavior(p, "boundary_pushing") or has_behavior(p, "no_not_respected"),
        reason="Boundary pushing / 'no' not respected is strong abnormality."
    ))

    rules.append(Rule(
        rule_id="HIGH_ASKS_YOU_TO_REJECT",
        severity="HIGH",
        abnormality=7,
        tags=["avoidance", "cowardice"],
        condition=lambda p, nn: phrase_matches(p, [r"can you reject me"]),
        reason="Asking you to reject him so he avoids telling his parents indicates low accountability."
    ))

    rules.append(Rule(
        rule_id="HIGH_RELIGIOUS_DEFLECTION",
        severity="HIGH",
        abnormality=6,
        tags=["performance", "integrity"],
        condition=lambda p, nn: has_behavior(p, "religious_deflection") or phrase_matches(p, [r"pravachan", r"family values.*", r"religio"]),
        reason="Over-preaching/pravachan in response to hard questions can signal deflection/performance."
    ))

    rules.append(Rule(
        rule_id="HIGH_FINANCE_UTILITY_WIFE_CHECKLIST",
        severity="HIGH",
        abnormality=6,
        tags=["utility-view", "control"],
        condition=lambda p, nn: has_behavior(p, "wife_checklist") or phrase_matches(p, [r"driving", r"skills (a )?woman should have", r"finances.*important"]),
        reason="Treating spouse as utility checklist (skills/finances) can be a control/transactional mindset."
    ))

    # ---------------- MEDIUM RISK ----------------
    rules.append(Rule(
        rule_id="MED_LOVE_BOMBING",
        severity="MEDIUM",
        abnormality=5,
        tags=["watch", "soft-risk"],
        condition=lambda p, nn: has_behavior(p, "love_bombing") or has_behavior(p, "compliment_bombing"),
        reason="Love/compliment bombing early can be manipulation. Watch for boundary tests."
    ))

    rules.append(Rule(
        rule_id="MED_DIVORCE_HYPOTHETICALS_EARLY",
        severity="MEDIUM",
        abnormality=5,
        tags=["watch", "soft-risk"],
        condition=lambda p, nn: phrase_matches(p, [r"divorce", r"case scenarios", r"std", r"sti"]),
        reason="Obsessive divorce/STD hypotheticals early can signal insecurity/control framing."
    ))

    rules.append(Rule(
        rule_id="MED_EX_TALK_EARLY",
        severity="MEDIUM",
        abnormality=4,
        tags=["watch", "soft-risk"],
        condition=lambda p, nn: phrase_matches(p, [r"\bmy ex\b", r"she('d| would) be my wife"]),
        reason="Ex-centric talk early is a watch-item for unresolved attachment."
    ))

    rules.append(Rule(
        rule_id="MED_INCOHERENT_VALUES",
        severity="MEDIUM",
        abnormality=5,
        tags=["inconsistency", "integrity"],
        condition=lambda p, nn: has_behavior(p, "contradictory_story") or has_behavior(p, "timeline_inconsistent"),
        reason="Contradictions/inconsistent story indicate unreliability or deception."
    ))

    rules.append(Rule(
        rule_id="MED_DISMISSIVE_NONCHALANT_MARRIAGE",
        severity="MEDIUM",
        abnormality=4,
        tags=["low-intent", "immaturity"],
        condition=lambda p, nn: phrase_matches(p, [r"you('ll| will) get used to it", r"been through many of this"]) or has_behavior(p, "nonchalant_about_marriage"),
        reason="Dismissive/nonchalant attitude toward marriage suggests low intent or immaturity."
    ))

    # ---------------- LOW/INFO (verification) ----------------
    rules.append(Rule(
        rule_id="INFO_UNKNOWN_KEY_NONNEGS",
        severity="INFO",
        abnormality=2,
        tags=["verify-nonneg", "watch"],
        condition=lambda p, nn: (
            (nn.partner_must_be_virgin and p.is_virgin is None) or
            (nn.no_smoking and p.smoked is None) or
            (nn.no_addictions and (p.drinks is None and p.drugs is None)) or
            (nn.must_be_vegetarian and p.vegetarian is None) or
            (nn.must_be_brahmin and p.brahmin is None) or
            (nn.must_be_south_indian and p.south_indian is None)
        ),
        reason="Missing info on key non-negotiables—verify early (not a red flag by itself)."
    ))

    return rules


# ============================================================
# 7) Pretty printing + demo
# ============================================================

def pretty_print(dec: Decision, p: Prospect) -> None:
    print("=" * 88)
    print(f"{p.id} | {p.alias}")
    print(f"Decision: {dec.decision} | Category: {dec.category}")
    print(f"Abnormality total: {dec.abnormality_total} ({dec.abnormality_level})")
    print(f"Non-neg mismatch: {dec.nonneg_mismatch}")
    if dec.unknown_nonneg_fields:
        print("Unknown non-negs to verify:", ", ".join(dec.unknown_nonneg_fields))
    print("Tags:", ", ".join(dec.tags) if dec.tags else "(none)")

    print("\nFired rules (each 1..10 abnormality):")
    if not dec.fired_rules:
        print(" - (none)")
    else:
        for h in sorted(dec.fired_rules, key=lambda x: (-x.abnormality, x.rule_id)):
            print(f" - {h.rule_id:32s}  {h.abnormality:2d}/10 [{h.severity}] :: {h.reason}")


def demo_inputs() -> List[Prospect]:
    return [
        Prospect(
            id="P1",
            alias="Angry My Way Highway Sherlyn Chopra Fan",
            key_phrases=["waiting for women in a queue", "I have a lot of kopam"],
            behaviors={"entitlement", "anger"},
            notes="Irritated, ego-driven vibe.",
            attraction="attracted",
            age_diff_years=3,
            vegetarian=True,
            brahmin=True,
            south_indian=True,
            is_virgin=None
        ),
        Prospect(
            id="P2",
            alias="Nonchalant Smoker",
            key_phrases=["Have been through many of this, you’ll get used to it"],
            behaviors={"nonchalant_about_marriage"},
            smoked=True,
            is_virgin=False,
            attraction="attracted",
            age_diff_years=3,
            vegetarian=False,
            brahmin=True,
            south_indian=True
        ),
        Prospect(
            id="P6",
            alias="Ghosting Performing-Empathy Catfisher",
            key_phrases=[
                "let’s see where this goes if it has to happen it will",
                "I’m not telling anything to my kids about what I did",
                "alcohol? Of course Bangalore",
            ],
            behaviors={"ghosting", "returned_after_ghosting", "catfishing", "age_manipulation", "help_as_leverage"},
            notes="Changed age, targeted 18–33 bracket.",
            attraction="attracted",
            age_diff_years=5,
            vegetarian=True,
            brahmin=True,
            south_indian=True,
            is_virgin=False
        ),
        Prospect(
            id="d9",
            alias="Stalking Plant Scientist",
            behaviors={"stalking", "boundary_pushing"},
            notes="Repeated unwanted views and forced connection; safety fear.",
            attraction="not_attracted",
            age_diff_years=4
        ),
        Prospect(
            id="d15",
            alias="Mocking Projector All Women Are Bad Guy",
            behaviors={"misogyny"},
            key_phrases=["all women are bad", "women are evil"],
            notes="Mocked values; projected hatred.",
            attraction="not_attracted",
            age_diff_years=4
        ),
        Prospect(
            id="P7",
            alias="Seemed Normal (but mismatch example)",
            key_phrases=[
                "this is not a random conversation",
                "I propose we take time and decide if we both want to talk again",
                "growth mentality is something I’m looking for"
            ],
            behaviors={"direct", "reflective", "respected_no"},
            is_virgin=False,         # mismatch for your NN
            smoked=False,
            atheist=False,
            wants_kids=True,
            vegetarian=True,
            brahmin=True,
            south_indian=True,
            attraction="attracted",
            age_diff_years=0
        ),
    ]


def main():
    nn = NonNegotiables(
        partner_must_be_virgin=True,
        no_smoking=True,
        no_addictions=True,
        wants_kids_required=True,
        not_atheist=True,
        must_be_vegetarian=True,
        must_be_brahmin=True,
        must_be_south_indian=True,
        zero_tolerance_stalking=True,
        zero_tolerance_catfishing=True,
        zero_tolerance_sexual_coercion=True,
        zero_tolerance_misogyny=True,
        max_age_diff_years=2
    )

    engine = WeedOutEngine(
        rules=build_rules(),
        not_normal_threshold=18,
        unsafe_threshold=26
    )

    for p in demo_inputs():
        dec = engine.evaluate(p, nn)
        pretty_print(dec, p)


if __name__ == "__main__":
    main()


P1 | Angry My Way Highway Sherlyn Chopra Fan
Decision: REJECT | Category: NORMAL_MISMATCH
Abnormality total: 10 (medium)
Non-neg mismatch: True
Unknown non-negs to verify: is_virgin, smoked, drinks/drugs, wants_kids, atheist
Tags: entitlement, objectification, verify-nonneg, watch

Fired rules (each 1..10 abnormality):
 - HIGH_ENTITLEMENT_QUEUE_LANGUAGE    8/10 [HIGH] :: Entitlement/objectification language about women is a strong character red flag.
 - INFO_UNKNOWN_KEY_NONNEGS           2/10 [INFO] :: Missing info on key non-negotiables—verify early (not a red flag by itself).
P2 | Nonchalant Smoker
Decision: REJECT | Category: NORMAL_MISMATCH
Abnormality total: 6 (low)
Non-neg mismatch: True
Unknown non-negs to verify: drinks/drugs, wants_kids, atheist
Tags: immaturity, low-intent, verify-nonneg, watch

Fired rules (each 1..10 abnormality):
 - MED_DISMISSIVE_NONCHALANT_MARRIAGE   4/10 [MEDIUM] :: Dismissive/nonchalant attitude toward marriage suggests low intent or immaturity.
 - INF

below is legacy

In [2]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable, Dict, List, Optional, Tuple, Set
import re

# ============================================================
# 1) Your Non-Negotiables (edit here)
# ============================================================

@dataclass
class NonNegotiables:
    # Core values
    partner_must_be_virgin: bool = True
    no_smoking: bool = True
    no_addictions: bool = True  # alcohol/drugs/gambling etc (you can refine)
    wants_kids_required: bool = True  # if you require "wants kids"
    not_atheist: bool = True  # if atheism is a hard mismatch

    # Safety/Behavioral hard lines
    zero_tolerance_stalking: bool = True
    zero_tolerance_catfishing: bool = True
    zero_tolerance_sexual_coercion: bool = True
    zero_tolerance_misogyny: bool = True

    # Optional preferences / constraints (not always hard)
    max_age_diff_years: Optional[int] = 2  # you can set None if not using
    prefer_same_birth_year: bool = False   # set True if you want to enforce 1990-only etc


# ============================================================
# 2) Data model for each man
# ============================================================

@dataclass
class Prospect:
    id: str
    alias: str = ""

    # Observations / evidence
    key_phrases: List[str] = field(default_factory=list)  # exact-ish quotes
    notes: str = ""                                       # your summary notes
    behaviors: Set[str] = field(default_factory=set)      # normalized tokens
    # examples: {"ghosting", "stalking", "catfishing", "asked_for_nudes",
    #            "dodged_nonneg", "compliment_bombing", "love_bombing",
    #            "misogyny", "help_as_leverage", "hot_cold", "identity_mismatch"}

    # Facts/claims
    is_virgin: Optional[bool] = None
    smoked: Optional[bool] = None
    drinks: Optional[bool] = None
    drugs: Optional[bool] = None
    wants_kids: Optional[bool] = None
    atheist: Optional[bool] = None
    age_diff_years: Optional[int] = None

    # your attraction (not used for safety gating, but kept for reporting)
    attraction: Optional[str] = None  # "attracted", "not_attracted", None


# ============================================================
# 3) Rule Engine
# ============================================================

@dataclass
class RuleHit:
    rule_id: str
    points: int
    severity: str       # "HARD_EXIT" | "HIGH" | "MEDIUM" | "LOW" | "GREEN"
    tags: List[str]
    reason: str


@dataclass
class Decision:
    category: str       # UNSAFE | NOT_NORMAL | NORMAL_MISMATCH | NORMAL_POSSIBLE
    decision: str       # BLOCK | REJECT | CAUTION | PASS
    risk_score: int
    nonneg_mismatch: bool
    tags: List[str]
    fired_rules: List[RuleHit]


@dataclass
class Rule:
    rule_id: str
    severity: str
    points: int
    tags: List[str]
    condition: Callable[[Prospect, NonNegotiables], bool]
    reason: str


class WeedOutEngine:
    """
    Gating philosophy:
    - HARD_EXIT rules => immediate BLOCK (UNSAFE)
    - Otherwise accumulate risk points.
    - Separately compute "nonneg mismatch" (NORMAL_MISMATCH vs NORMAL_POSSIBLE).
    """

    def __init__(
        self,
        rules: List[Rule],
        hard_exit_severity: str = "HARD_EXIT",
        not_normal_threshold: int = 8,
        unsafe_threshold: int = 14
    ):
        self.rules = rules
        self.hard_exit_severity = hard_exit_severity
        self.not_normal_threshold = not_normal_threshold
        self.unsafe_threshold = unsafe_threshold

    def evaluate(self, p: Prospect, nn: NonNegotiables) -> Decision:
        hits: List[RuleHit] = []
        tags: Set[str] = set()
        score = 0

        # Evaluate rules
        for r in self.rules:
            if r.condition(p, nn):
                score += r.points
                tags.update(r.tags)
                hits.append(RuleHit(
                    rule_id=r.rule_id,
                    points=r.points,
                    severity=r.severity,
                    tags=r.tags,
                    reason=r.reason
                ))

        # HARD EXIT gate
        if any(h.severity == self.hard_exit_severity for h in hits):
            return Decision(
                category="UNSAFE",
                decision="BLOCK",
                risk_score=max(score, self.unsafe_threshold),
                nonneg_mismatch=True,  # irrelevant; safety dominates
                tags=sorted(tags),
                fired_rules=hits
            )

        # Compute non-neg mismatch (separate from "not normal")
        mismatch = self._nonneg_mismatch(p, nn)

        # Determine category based on risk score
        if score >= self.unsafe_threshold:
            category = "UNSAFE"
            decision_str = "BLOCK"
        elif score >= self.not_normal_threshold:
            category = "NOT_NORMAL"
            decision_str = "REJECT"
        else:
            # risk low => then it becomes about alignment
            if mismatch:
                category = "NORMAL_MISMATCH"
                decision_str = "REJECT"
            else:
                category = "NORMAL_POSSIBLE"
                decision_str = "PASS"

        # If low-ish risk but there are watch-tags, downgrade to CAUTION
        if decision_str == "PASS" and any(t in tags for t in ["watch", "soft-risk", "inconsistency"]):
            decision_str = "CAUTION"

        return Decision(
            category=category,
            decision=decision_str,
            risk_score=score,
            nonneg_mismatch=mismatch,
            tags=sorted(tags),
            fired_rules=hits
        )

    def _nonneg_mismatch(self, p: Prospect, nn: NonNegotiables) -> bool:
        # Only count mismatch if we actually have a signal
        # (unknown is not mismatch; it’s "needs verification")
        if nn.partner_must_be_virgin and p.is_virgin is False:
            return True
        if nn.no_smoking and p.smoked is True:
            return True
        if nn.no_addictions:
            if p.drinks is True or p.drugs is True:
                return True
        if nn.wants_kids_required and p.wants_kids is False:
            return True
        if nn.not_atheist and p.atheist is True:
            return True
        if nn.max_age_diff_years is not None and p.age_diff_years is not None:
            if abs(p.age_diff_years) > nn.max_age_diff_years:
                return True
        return False


# ============================================================
# 4) Helpers for phrase/behavior matching
# ============================================================

def has_behavior(p: Prospect, *tokens: str) -> bool:
    lower = {b.lower() for b in p.behaviors}
    return any(t.lower() in lower for t in tokens)

def phrase_matches(p: Prospect, patterns: List[str]) -> bool:
    text = " | ".join(p.key_phrases + [p.notes]).lower()
    return any(re.search(ptn, text) for ptn in patterns)


# ============================================================
# 5) The Rule Set (customized for your logs + your red flags)
# ============================================================

def build_rules() -> List[Rule]:
    rules: List[Rule] = []

    # ---------- HARD EXIT (UNSAFE) ----------
    rules.append(Rule(
        rule_id="HARD_STALKING",
        severity="HARD_EXIT",
        points=20,
        tags=["safety", "stalking"],
        condition=lambda p, nn: nn.zero_tolerance_stalking and has_behavior(p, "stalking", "doxxing"),
        reason="Stalking/doxxing is a hard safety exit."
    ))

    rules.append(Rule(
        rule_id="HARD_CATFISHING_IDENTITY",
        severity="HARD_EXIT",
        points=20,
        tags=["safety", "deception", "catfishing"],
        condition=lambda p, nn: nn.zero_tolerance_catfishing and has_behavior(p, "catfishing", "identity_mismatch", "age_manipulation"),
        reason="Identity/age manipulation is a hard safety exit."
    ))

    rules.append(Rule(
        rule_id="HARD_SEXUAL_COERCION",
        severity="HARD_EXIT",
        points=20,
        tags=["safety", "sexual-coercion"],
        condition=lambda p, nn: nn.zero_tolerance_sexual_coercion and has_behavior(p, "asked_for_nudes", "sexual_pressure"),
        reason="Sexual coercion / nudes requests early is a hard exit."
    ))

    rules.append(Rule(
        rule_id="HARD_MISOGYNY",
        severity="HARD_EXIT",
        points=20,
        tags=["safety", "misogyny"],
        condition=lambda p, nn: nn.zero_tolerance_misogyny and has_behavior(p, "misogyny"),
        reason="Misogyny is unsafe and a hard exit."
    ))

    # ---------- HIGH RISK (NOT NORMAL) ----------
    rules.append(Rule(
        rule_id="HIGH_GHOSTING_PATTERN",
        severity="HIGH",
        points=8,
        tags=["reliability", "inconsistency"],
        condition=lambda p, nn: has_behavior(p, "ghosting") and has_behavior(p, "returned_after_ghosting"),
        reason="Ghosting + returning indicates hot-cold cycles and unreliability."
    ))

    rules.append(Rule(
        rule_id="HIGH_GHOSTING_SINGLE",
        severity="HIGH",
        points=6,
        tags=["reliability", "inconsistency"],
        condition=lambda p, nn: has_behavior(p, "ghosting"),
        reason="Ghosting indicates low reliability and poor communication."
    ))

    rules.append(Rule(
        rule_id="HIGH_DODGE_NONNEG",
        severity="HIGH",
        points=7,
        tags=["integrity", "nonneg-evasion"],
        condition=lambda p, nn: has_behavior(p, "dodged_nonneg") or phrase_matches(p, [r"\bavoid(ed)?\b", r"\bdeflect(ed)?\b", r"\bwent silent\b"]),
        reason="Dodging direct non-negotiable questions is a common deception pattern."
    ))

    rules.append(Rule(
        rule_id="HIGH_HELP_AS_LEVERAGE",
        severity="HIGH",
        points=6,
        tags=["power-dynamic", "control"],
        condition=lambda p, nn: has_behavior(p, "help_as_leverage") or phrase_matches(p, [r"helping because you('re| are) a girl", r"you need help"]),
        reason="Help framed through gender/power can become leverage and control."
    ))

    rules.append(Rule(
        rule_id="HIGH_ENTITLEMENT_QUEUE",
        severity="HIGH",
        points=8,
        tags=["entitlement", "objectification"],
        condition=lambda p, nn: phrase_matches(p, [r"queue", r"line of girls", r"waiting for women"]),
        reason="Entitlement language about women is a strong character red flag."
    ))

    rules.append(Rule(
        rule_id="HIGH_ASKS_YOU_TO_REJECT",
        severity="HIGH",
        points=6,
        tags=["avoidance", "cowardice"],
        condition=lambda p, nn: phrase_matches(p, [r"can you reject me"]),
        reason="Asking you to reject him so he avoids accountability is maturity failure."
    ))

    rules.append(Rule(
        rule_id="HIGH_RELIGIOUS_PRAVACHAN_DEFLECTION",
        severity="HIGH",
        points=6,
        tags=["performance", "integrity"],
        condition=lambda p, nn: phrase_matches(p, [r"pravachan", r"family values", r"\breligio"]),
        reason="Over-preaching during hard questions can be a cover story (watch for deflection)."
    ))

    # ---------- MEDIUM RISK ----------
    rules.append(Rule(
        rule_id="MED_COMPLIMENT_BOMBING",
        severity="MEDIUM",
        points=4,
        tags=["soft-risk", "watch"],
        condition=lambda p, nn: has_behavior(p, "compliment_bombing") or has_behavior(p, "love_bombing"),
        reason="Love/compliment bombing early can be manipulation; watch for boundaries being tested."
    ))

    rules.append(Rule(
        rule_id="MED_DIVORCE_HYPOTHETICALS_EARLY",
        severity="MEDIUM",
        points=4,
        tags=["soft-risk", "watch"],
        condition=lambda p, nn: phrase_matches(p, [r"divorce", r"case scenarios", r"std", r"sti"]) and not has_behavior(p, "mature_context"),
        reason="Obsessive divorce hypotheticals early can signal insecurity/control framing."
    ))

    rules.append(Rule(
        rule_id="MED_EX_TALK_EARLY",
        severity="MEDIUM",
        points=3,
        tags=["soft-risk", "watch"],
        condition=lambda p, nn: phrase_matches(p, [r"\bmy ex\b", r"she('d| would) be my wife"]),
        reason="Ex-centric talk early isn’t always bad, but it’s a watch-item for unresolved attachment."
    ))

    # ---------- GREEN FLAGS (reduce score) ----------
    rules.append(Rule(
        rule_id="GREEN_DIRECT_CALM",
        severity="GREEN",
        points=-3,
        tags=["green", "eq"],
        condition=lambda p, nn: phrase_matches(p, [r"let'?s take time", r"not a random conversation", r"i propose we take time"]),
        reason="Slowing down + directness often indicates emotional regulation."
    ))

    rules.append(Rule(
        rule_id="GREEN_RESPECTS_BOUNDARY",
        severity="GREEN",
        points=-3,
        tags=["green", "respect"],
        condition=lambda p, nn: has_behavior(p, "respected_no") and not has_behavior(p, "boundary_pushing"),
        reason="Respects 'no' without bargaining—strong normal marker."
    ))

    return rules


# ============================================================
# 6) Example: Add your men as inputs (you can extend this list)
# ============================================================

def demo_inputs() -> List[Prospect]:
    return [
        Prospect(
            id="P1",
            alias="Angry My Way Highway Sherlyn Chopra Fan",
            key_phrases=["waiting for women in a queue", "I have a lot of kopam"],
            behaviors={"entitlement", "anger"},
            notes="Irritated, ego-driven vibe.",
            attraction="attracted",
            age_diff_years=3
        ),
        Prospect(
            id="P2",
            alias="Nonchalant Smoker",
            key_phrases=["Have been through many of this, you’ll get used to it"],
            behaviors={"nonchalant"},
            smoked=True,
            is_virgin=False,
            attraction="attracted",
            age_diff_years=3
        ),
        Prospect(
            id="P6",
            alias="Ghosting Performing-Empathy Catfisher",
            key_phrases=[
                "let’s see where this goes if it has to happen it will",
                "I’m not telling anything to my kids about what I did",
                "alcohol? Of course Bangalore",
            ],
            behaviors={"ghosting", "returned_after_ghosting", "catfishing", "age_manipulation", "help_as_leverage"},
            notes="Changed age, targeted 18–33 bracket.",
            attraction="attracted",
            age_diff_years=5
        ),
        Prospect(
            id="d9",
            alias="Stalking Plant Scientist",
            behaviors={"stalking", "boundary_pushing"},
            notes="Repeated unwanted views and forced connection; safety fear.",
            attraction="not_attracted",
            age_diff_years=4
        ),
        Prospect(
            id="d15",
            alias="Mocking Projector All Women Are Bad Guy",
            behaviors={"misogyny"},
            key_phrases=["all women are bad", "women are evil"],
            notes="Mocked values; projected hatred.",
            attraction="not_attracted",
            age_diff_years=4
        ),
        Prospect(
            id="P7",
            alias="Seemed Normal",
            key_phrases=[
                "this is not a random conversation",
                "I propose we take time and decide if we both want to talk again",
                "growth mentality is something I’m looking for"
            ],
            behaviors={"direct", "reflective", "respected_no"},
            is_virgin=False,  # (example) if not aligned with your non-neg
            smoked=False,
            atheist=False,
            wants_kids=True,
            attraction="attracted",
            age_diff_years=0
        ),
    ]


# ============================================================
# 7) Run it
# ============================================================

def pretty_print(dec: Decision, p: Prospect) -> None:
    print("=" * 80)
    print(f"{p.id} | {p.alias}")
    print(f"Decision: {dec.decision} | Category: {dec.category} | Risk: {dec.risk_score} | NonNegMismatch: {dec.nonneg_mismatch}")
    print("Tags:", ", ".join(dec.tags) if dec.tags else "(none)")
    print("\nFired rules:")
    for h in sorted(dec.fired_rules, key=lambda x: (x.severity != "HARD_EXIT", -x.points)):
        print(f" - {h.rule_id:30s} {h.points:+3d} [{h.severity}] :: {h.reason}")


def main():
    nn = NonNegotiables(
        partner_must_be_virgin=True,
        no_smoking=True,
        no_addictions=True,
        wants_kids_required=True,
        not_atheist=True,
        zero_tolerance_stalking=True,
        zero_tolerance_catfishing=True,
        zero_tolerance_sexual_coercion=True,
        zero_tolerance_misogyny=True,
        max_age_diff_years=2
    )

    engine = WeedOutEngine(
        rules=build_rules(),
        not_normal_threshold=8,
        unsafe_threshold=14
    )

    prospects = demo_inputs()

    for p in prospects:
        dec = engine.evaluate(p, nn)
        pretty_print(dec, p)


if __name__ == "__main__":
    main()


P1 | Angry My Way Highway Sherlyn Chopra Fan
Decision: REJECT | Category: NOT_NORMAL | Risk: 8 | NonNegMismatch: True
Tags: entitlement, objectification

Fired rules:
 - HIGH_ENTITLEMENT_QUEUE          +8 [HIGH] :: Entitlement language about women is a strong character red flag.
P2 | Nonchalant Smoker
Decision: REJECT | Category: NORMAL_MISMATCH | Risk: 0 | NonNegMismatch: True
Tags: (none)

Fired rules:
P6 | Ghosting Performing-Empathy Catfisher
Decision: BLOCK | Category: UNSAFE | Risk: 40 | NonNegMismatch: True
Tags: catfishing, control, deception, inconsistency, power-dynamic, reliability, safety

Fired rules:
 - HARD_CATFISHING_IDENTITY       +20 [HARD_EXIT] :: Identity/age manipulation is a hard safety exit.
 - HIGH_GHOSTING_PATTERN           +8 [HIGH] :: Ghosting + returning indicates hot-cold cycles and unreliability.
 - HIGH_GHOSTING_SINGLE            +6 [HIGH] :: Ghosting indicates low reliability and poor communication.
 - HIGH_HELP_AS_LEVERAGE           +6 [HIGH] :: Help fr

In [3]:
@dataclass
class NonNegotiables:
    # Core values
    partner_must_be_virgin: bool = True
    no_smoking: bool = True
    no_addictions: bool = True
    wants_kids_required: bool = True
    not_atheist: bool = True

    # NEW: culture/food non-negs
    must_be_vegetarian: bool = True
    must_be_brahmin: bool = True
    must_be_south_indian: bool = True   # or set False if you only care about Brahmin

    # Safety/Behavioral hard lines
    zero_tolerance_stalking: bool = True
    zero_tolerance_catfishing: bool = True
    zero_tolerance_sexual_coercion: bool = True
    zero_tolerance_misogyny: bool = True

    # Optional constraints
    max_age_diff_years: Optional[int] = 2
    prefer_same_birth_year: bool = False


In [4]:
@dataclass
class Prospect:
    id: str
    alias: str = ""

    key_phrases: List[str] = field(default_factory=list)
    notes: str = ""
    behaviors: Set[str] = field(default_factory=set)

    # Facts/claims
    is_virgin: Optional[bool] = None
    smoked: Optional[bool] = None
    drinks: Optional[bool] = None
    drugs: Optional[bool] = None
    wants_kids: Optional[bool] = None
    atheist: Optional[bool] = None
    age_diff_years: Optional[int] = None

    # NEW: veg + community
    vegetarian: Optional[bool] = None
    brahmin: Optional[bool] = None
    south_indian: Optional[bool] = None  # True/False/None

    attraction: Optional[str] = None


In [5]:
def _nonneg_mismatch(self, p: Prospect, nn: NonNegotiables) -> bool:
    if nn.partner_must_be_virgin and p.is_virgin is False:
        return True
    if nn.no_smoking and p.smoked is True:
        return True
    if nn.no_addictions and (p.drinks is True or p.drugs is True):
        return True
    if nn.wants_kids_required and p.wants_kids is False:
        return True
    if nn.not_atheist and p.atheist is True:
        return True

    # NEW: vegetarian + Brahmin + South Indian
    if nn.must_be_vegetarian and p.vegetarian is False:
        return True
    if nn.must_be_brahmin and p.brahmin is False:
        return True
    if nn.must_be_south_indian and p.south_indian is False:
        return True

    if nn.max_age_diff_years is not None and p.age_diff_years is not None:
        if abs(p.age_diff_years) > nn.max_age_diff_years:
            return True

    return False


In [6]:
def _nonneg_mismatch(self, p: Prospect, nn: NonNegotiables) -> bool:
    if nn.partner_must_be_virgin and p.is_virgin is False:
        return True
    if nn.no_smoking and p.smoked is True:
        return True
    if nn.no_addictions and (p.drinks is True or p.drugs is True):
        return True
    if nn.wants_kids_required and p.wants_kids is False:
        return True
    if nn.not_atheist and p.atheist is True:
        return True

    # NEW: vegetarian + Brahmin + South Indian
    if nn.must_be_vegetarian and p.vegetarian is False:
        return True
    if nn.must_be_brahmin and p.brahmin is False:
        return True
    if nn.must_be_south_indian and p.south_indian is False:
        return True

    if nn.max_age_diff_years is not None and p.age_diff_years is not None:
        if abs(p.age_diff_years) > nn.max_age_diff_years:
            return True

    return False


In [8]:
rules.append(Rule(
    rule_id="WATCH_UNKNOWN_VEG_BRAHMIN_SOUTH",
    severity="MEDIUM",
    points=2,
    tags=["watch", "verify-nonneg"],
    condition=lambda p, nn: (
        (nn.must_be_vegetarian and p.vegetarian is None) or
        (nn.must_be_brahmin and p.brahmin is None) or
        (nn.must_be_south_indian and p.south_indian is None)
    ),
    reason="Key non-negotiables (veg/Brahmin/South Indian) are unknown — verify early."
))


NameError: name 'rules' is not defined

In [9]:
p = Prospect(
    id="d18",
    alias="Younger Guy",
    vegetarian=False,   # eats chicken
    brahmin=True,
    south_indian=True,
    behaviors={"direct", "respected_no"},
)
